# 02 — SfM-30k data, DINOv2 smoke checks, pooling pilot, and feature cache

Run this notebook before a full cache. It validates token layout and preprocessing, runs the SfM-only pooling-temperature pilot, then calls the resumable cache script after a temperature is locked.

In [ ]:
%pip install -q -e .
from pathlib import Path
import torch

from cbir.backbone import FrozenDinoV2Extractor
from cbir.config import load_project_config
from cbir.data.sfm import Sfm30kMetadata, SfmImageDirectoryReader, SfmMatImageReader
from cbir.features import FeatureExtractionRunner
from cbir.plotting import SeriesData, plot_series

CONFIG_PATH = Path('configs/extraction_sfm30k.yaml')
cfg = load_project_config(CONFIG_PATH)
print(cfg)

## Phase 0: square/rectangular token smoke test

This must show a 16×16 patch grid for 224×224 and a 16×10 grid for 224×140. The official intermediate API should return genuine patches only; the adapter asserts the register-token convention.

In [ ]:
extractor = FrozenDinoV2Extractor(cfg.backbone)
for hw in [(224, 224), (224, 140)]:
    x = torch.randn(1, 3, *hw)
    outputs = extractor.extract_intermediate_tokens(x, (3, 7, 11))
    print(hw, [(item.block_index, item.patch_grid_hw, tuple(item.patches.shape)) for item in outputs])

## Phase 0b: end-to-end integration smoke run

This is deliberately a tiny non-benchmark run. It uses real DINOv2 on generated images, writes and reads a temporary feature cache, trains the RGMF head for two epochs, reloads its checkpoint, and runs SfM-style plus synthetic RevisitOP evaluation. Its purpose is to catch interface, cache, device, and checkpoint mistakes before downloading or processing SfM-30k. Its scores have no scientific meaning.

In [ ]:
from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory

from PIL import Image, ImageDraw

from cbir.cache import FeatureManifest, FeatureShardReader, FeatureShardWriter, validate_feature_cache
from cbir.config import FeatureCacheConfig, FusionConfig, TrainingConfig
from cbir.data.revisitop import RevisitGroundTruth, crop_revisit_query
from cbir.data.sfm import ImageRecord, PairRecord, ValidationCase
from cbir.evaluation import (
    descriptors_from_cache,
    descriptors_from_images,
    evaluate_revisitop,
    evaluate_sfm_verified_pairs,
    ranked_ids_from_descriptors,
)
from cbir.fusion import ReliabilityGatedFusion
from cbir.training import HeadTrainer
from cbir.utils import hash_strings

def toy_landmark(color, shift=0):
    image = Image.new('RGB', (190, 130), (238, 238, 238))
    draw = ImageDraw.Draw(image)
    draw.rectangle((35 + shift, 20, 145 + shift, 112), fill=color, outline=(20, 20, 20), width=4)
    draw.ellipse((75 + shift, 45, 110 + shift, 80), fill=(250, 250, 250))
    return image

def toy_query(color):
    image = Image.new('RGB', (240, 160), (245, 245, 245))
    draw = ImageDraw.Draw(image)
    draw.rectangle((55, 32, 180, 138), fill=color, outline=(20, 20, 20), width=4)
    draw.ellipse((100, 65, 138, 103), fill=(250, 250, 250))
    return image, (55, 32, 180, 138)

# Two train clusters and two disjoint validation clusters; one confirmed pair per cluster.
specs = [
    ('train_red_q', (205, 45, 45), 0, 'train', 0),
    ('train_red_p', (205, 45, 45), 0, 'train', 4),
    ('train_blue_q', (45, 70, 205), 1, 'train', 0),
    ('train_blue_p', (45, 70, 205), 1, 'train', -4),
    ('val_green_q', (45, 160, 75), 2, 'val', 0),
    ('val_green_p', (45, 160, 75), 2, 'val', 4),
    ('val_gold_q', (205, 160, 35), 3, 'val', 0),
    ('val_gold_p', (205, 160, 35), 3, 'val', -4),
]
smoke_images = [(image_id, toy_landmark(color, shift)) for image_id, color, _, _, shift in specs]
smoke_records = {
    image_id: ImageRecord(image_id, split=split, cluster_id=cluster)
    for image_id, _, cluster, split, _ in specs
}
smoke_train_pairs = (
    PairRecord('train_red_q', 'train_red_p', 0, 'train'),
    PairRecord('train_blue_q', 'train_blue_p', 1, 'train'),
)
smoke_val_ids = ('val_green_q', 'val_green_p', 'val_gold_q', 'val_gold_p')
smoke_val_cases = (
    ValidationCase('val_green_q', 'val_green_p', 2, frozenset({'val_green_q'})),
    ValidationCase('val_gold_q', 'val_gold_p', 3, frozenset({'val_gold_q'})),
)

runner = FeatureExtractionRunner(extractor, cfg.preprocess, cfg.pooling)
smoke_features = runner.extract_images(
    smoke_images,
    layer_indices=cfg.pooling.all_layer_indices,
    backbone_batch_size=4,
)

with TemporaryDirectory(prefix='cbir-smoke-') as temporary_name:
    cache_dir = Path(temporary_name) / 'cache'
    smoke_ids = tuple(image_id for image_id, _ in smoke_images)
    manifest = FeatureManifest(
        fingerprint='integration-smoke-v1',
        dataset_name='generated-smoke',
        source_ids_hash=hash_strings(smoke_ids),
        extraction_config={'purpose': 'integration-smoke'},
        layer_indices=cfg.pooling.all_layer_indices,
        token_dim=cfg.backbone.token_dim,
        feature_dtype='float16',
        entropy_dtype='float32',
        expected_image_ids=smoke_ids,
    )
    writer = FeatureShardWriter(
        cache_dir,
        manifest,
        FeatureCacheConfig(local_root=cache_dir, shard_size=len(smoke_ids)),
    )
    writer.write_shard(smoke_features, smoke_records)
    writer.finalize()
    assert validate_feature_cache(cache_dir)['valid']

    cache_reader = FeatureShardReader(cache_dir)
    smoke_fusion = FusionConfig(
        token_dim=cfg.backbone.token_dim,
        layer_indices=(3, 7, 11),
        output_dim=32,
        local_kind='cls_guided_patch',
        gate_mode='reliability',
    )
    smoke_training = TrainingConfig(
        seed=7,
        batch_size=2,
        epochs=2,
        learning_rate=1e-3,
        loss_temperature=0.07,
        early_stopping_patience=2,
        device=cfg.backbone.device,
    )
    head = ReliabilityGatedFusion.from_config(smoke_fusion)
    trainer = HeadTrainer(
        head=head,
        reader=cache_reader,
        train_pairs=smoke_train_pairs,
        fusion_config=smoke_fusion,
        training_config=smoke_training,
        validation_cases=smoke_val_cases,
        validation_image_ids=smoke_val_ids,
        output_dir=cache_dir / 'run',
    )
    history = trainer.fit()
    assert history.best_checkpoint is not None and history.best_checkpoint.is_file()

    checkpoint = torch.load(history.best_checkpoint, map_location='cpu', weights_only=False)
    reloaded_head = ReliabilityGatedFusion.from_config(smoke_fusion)
    reloaded_head.load_state_dict(checkpoint['model_state_dict'])
    val_descriptors = descriptors_from_cache(
        cache_reader,
        reloaded_head,
        smoke_val_ids,
        layer_indices=smoke_fusion.layer_indices,
        local_kind=smoke_fusion.local_kind,
        device=cfg.backbone.device,
    )
    sfm_report = evaluate_sfm_verified_pairs(val_descriptors, smoke_val_ids, smoke_val_cases)

    revisit_database_ids = ('rev_red_easy', 'rev_red_hard', 'rev_blue_easy', 'rev_blue_hard')
    revisit_database_images = [
        ('rev_red_easy', toy_landmark((205, 45, 45), 0)),
        ('rev_red_hard', toy_landmark((205, 45, 45), 5)),
        ('rev_blue_easy', toy_landmark((45, 70, 205), 0)),
        ('rev_blue_hard', toy_landmark((45, 70, 205), -5)),
    ]
    red_query, red_box = toy_query((205, 45, 45))
    blue_query, blue_box = toy_query((45, 70, 205))
    revisit_query_ids = ('rev_red_query', 'rev_blue_query')
    revisit_query_images = [
        ('rev_red_query', crop_revisit_query(red_query, red_box)),
        ('rev_blue_query', crop_revisit_query(blue_query, blue_box)),
    ]
    database_descriptors = descriptors_from_images(
        runner, reloaded_head, revisit_database_images,
        layer_indices=smoke_fusion.layer_indices,
        local_kind=smoke_fusion.local_kind,
        backbone_batch_size=4,
        device=cfg.backbone.device,
    )
    query_descriptors = descriptors_from_images(
        runner, reloaded_head, revisit_query_images,
        layer_indices=smoke_fusion.layer_indices,
        local_kind=smoke_fusion.local_kind,
        backbone_batch_size=2,
        device=cfg.backbone.device,
    )
    rankings = ranked_ids_from_descriptors(query_descriptors, database_descriptors, revisit_database_ids)
    revisit_report = evaluate_revisitop(
        rankings,
        revisit_query_ids,
        {
            'rev_red_query': RevisitGroundTruth('rev_red_query', frozenset({'rev_red_easy'}), frozenset({'rev_red_hard'}), frozenset()),
            'rev_blue_query': RevisitGroundTruth('rev_blue_query', frozenset({'rev_blue_easy'}), frozenset({'rev_blue_hard'}), frozenset()),
        },
    )
    assert 0.0 <= revisit_report.medium.map <= 1.0
    assert 0.0 <= revisit_report.hard.map <= 1.0

print('Integration smoke test passed.')
print('Toy SfM validation:', sfm_report)
print('Toy RevisitOP Medium/Hard mAP:', revisit_report.medium.map, revisit_report.hard.map)

## Metadata validation and pooling-temperature pilot

First run scripts/prepare_sfm30k.py in Colab to fetch metadata and the chosen image source. Select a deterministic 1–2k image sample spanning both splits. The pilot computes all candidate temperatures from the same backbone forwards; do not create the full cache until a value is documented and copied into the extraction configuration.

In [ ]:
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
print(metadata.validate())
records = list(metadata.records('train'))[:1000]

if cfg.sfm.image_root is not None:
    image_reader = SfmImageDirectoryReader(cfg.sfm.image_root)
else:
    image_reader = SfmMatImageReader(cfg.sfm.image_mat_path)
try:
    images = [
        (record.image_id, image_reader.read(record.image_id) if cfg.sfm.image_root is not None else image_reader.read(record.split, int(record.image_locator)))
        for record in records
    ]
finally:
    if isinstance(image_reader, SfmMatImageReader):
        image_reader.close()

runner = FeatureExtractionRunner(extractor, cfg.preprocess, cfg.pooling)
pilot = runner.pilot_pooling_temperatures(
    images,
    temperatures=(0.05, 0.1, 0.2, 0.5, 1.0, 2.0),
    backbone_batch_size=32,
)
series = {
    str(tau): SeriesData(x=list(range(len(pilot.layer_indices))), y=values.mean(dim=0).tolist())
    for tau, values in pilot.entropy_by_temperature.items()
}
plot_series(series, title='Mean normalized pooling entropy', xlabel='Layer position', ylabel='Entropy');

## Full resumable cache

After recording the chosen pooling temperature in the YAML config, first use the limit-500 command to test cache/Drive recovery. Then omit the limit option. The script checks local cache first, then Drive, and only computes unresolved features.

In [ ]:
# !python scripts/extract_features.py --config configs/extraction_sfm30k.yaml --limit 500
# !python scripts/extract_features.py --config configs/extraction_sfm30k.yaml --backbone-batch-size 32